# Can we predict `passed_quiz`?

**Question:** given what we know about a tutoring session, can we predict whether the student passes the quiz that closes it?

**Plan**
1. Check the data (structure, quality, leakage risks).
2. Fix an evaluation that matches the real use case. The data has many sessions per student, so a naive random split would flatter the model.
3. Compare simple baselines with a linear and a tree model.
4. Report how much signal there is, and what drives it.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, accuracy_score
from sklearn.inspection import permutation_importance

df = pd.read_csv('data.csv')
print(pd.read_csv('columns.csv').to_string(index=False))
df.head()

## 1. Data checks

In [ ]:
print(df.shape, '| missing:', int(df.isna().sum().sum()), '| duplicate rows:', int(df.duplicated().sum()))
print('students:', df.student_key.nunique(), '| sessions/student: median', df.groupby('student_key').size().median(),
      'max', df.groupby('student_key').size().max())
print('pass rate:', round(df.passed_quiz.mean(), 3))
df.describe().T

Each row is a session, not a student: ~1,800 students with ~7 sessions each. `enrol_year` and `school_code` are constant within a student, and `prior_score` varies slightly between a student's sessions (it is "as stored on the session record"; we treat it as a noisy measurement).

Is there student-level heterogeneity beyond what the session features explain? If students differ a lot in pass rate, rows from the same student are not independent.

In [ ]:
g = df.groupby('student_key').passed_quiz.agg(['mean', 'size'])
p = df.passed_quiz.mean()
print('variance of per-student pass rate, observed :', round(g['mean'].var(), 4))
print('expected if every student had the same rate :', round((p*(1-p)/g['size']).mean(), 4))
print('within-student std of prior_score:', round(df.groupby('student_key').prior_score.std().mean(), 2),
      '(overall std %.1f)' % df.prior_score.std())

Observed variance is about 3x what identical students would give, so **some students pass much more often than others**, for reasons not captured by the columns here. This matters for evaluation: a flexible model could pick up student identity (through `enrol_year` + `school_code` + `prior_score`) and look good on a random split without being useful for new students.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for a, c in zip(ax, ['session_minutes', 'hints_used', 'prior_score']):
    if c == 'hints_used':
        s = df.groupby(c).passed_quiz.agg(['mean', 'size']); s = s[s['size'] >= 30]
    else:
        s = df.groupby(pd.qcut(df[c], 10), observed=True).passed_quiz.agg(['mean', 'size'])
        s.index = [round(i.mid, 1) for i in s.index]
    a.plot(s.index, s['mean'], 'o-'); a.axhline(p, ls=':', c='gray')
    a.set_xlabel(c); a.set_ylabel('pass rate'); a.set_ylim(0.2, 0.65)
plt.tight_layout(); plt.show()
print(df.groupby('enrol_year').passed_quiz.mean().round(3).to_dict())

Longer sessions go with more passing, more hints with less passing (probably struggling students ask for more hints, so this is association, not an effect of hints). `prior_score` is nearly flat, which is surprising for a placement score.

## 2. Evaluation design

- **Metric:** ROC AUC (ranking quality, threshold-free), plus log loss / Brier, with a constant-rate baseline. Accuracy is shown but the classes are 42/58, so the majority baseline already scores 0.58.
- **Primary split: `GroupKFold` by student.** The question is "can we predict for a student we haven't seen?", so no student appears in both train and test.
- **For contrast:** a naive shuffled `KFold`, which lets the same student appear in train and test.

In [ ]:
X = df[['enrol_year', 'school_code', 'prior_score', 'session_minutes', 'hints_used']]
y = df.passed_quiz.values
groups = df.student_key.values
session_only = ['prior_score', 'session_minutes', 'hints_used']

def gb():
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)
def lr():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

models = {
    'constant rate': (DummyClassifier(strategy='prior'), list(X.columns)),
    'logistic (session features)': (lr(), session_only),
    'logistic (all)': (lr(), list(X.columns)),
    'boosted trees (session features)': (gb(), session_only),
    'boosted trees (all)': (gb(), list(X.columns)),
}

def evaluate(cv, **kw):
    rows, preds = [], {}
    for name, (m, cols) in models.items():
        pr = cross_val_predict(m, X[cols], y, cv=cv, method='predict_proba', **kw)[:, 1]
        preds[name] = pr
        rows.append(dict(model=name, AUC=roc_auc_score(y, pr), logloss=log_loss(y, pr),
                         brier=brier_score_loss(y, pr), accuracy=accuracy_score(y, pr > .5)))
    return pd.DataFrame(rows).set_index('model').round(4), preds

res_group, preds_group = evaluate(GroupKFold(n_splits=5), groups=groups)
res_group

### Same models under a naive shuffled split

In [ ]:
res_naive, _ = evaluate(KFold(n_splits=5, shuffle=True, random_state=0))
res_naive

## 3. Uncertainty and what drives the prediction

In [ ]:
# Bootstrap over students for the AUC of the best honest model
best = res_group.drop('constant rate').AUC.idxmax()
print('best grouped-CV model:', best)
pr = preds_group[best]
rng = np.random.default_rng(0)
ids = df.student_key.unique(); idx_by = df.groupby('student_key').indices
aucs = []
for _ in range(300):
    pick = rng.choice(ids, len(ids))
    ii = np.concatenate([idx_by[s] for s in pick])
    aucs.append(roc_auc_score(y[ii], pr[ii]))
print('AUC %.3f  (95%% CI over students: %.3f - %.3f)' % (roc_auc_score(y, pr), *np.percentile(aucs, [2.5, 97.5])))

In [ ]:
# Coefficients of the interpretable model (standardised features)
m = lr().fit(X[session_only], y)
print(pd.Series(m[-1].coef_[0], index=session_only).round(3).rename('log-odds per 1 SD'))

# Permutation importance on held-out students for the tree model (all features)
tr, te = next(GroupKFold(n_splits=5).split(X, y, groups))
mg = gb().fit(X.iloc[tr], y[tr])
pi = permutation_importance(mg, X.iloc[te], y[te], scoring='roc_auc', n_repeats=10, random_state=0)
pd.DataFrame({'AUC drop': pi.importances_mean, 'sd': pi.importances_std}, index=X.columns).round(4).sort_values('AUC drop', ascending=False)

In [ ]:
# Calibration of the best model under grouped CV
b = pd.qcut(pr, 10, duplicates='drop')
cal = pd.DataFrame({'pred': pr, 'obs': y}).groupby(b, observed=True).mean()
plt.figure(figsize=(4.2, 4)); plt.plot(cal.pred, cal.obs, 'o-'); plt.plot([.2, .7], [.2, .7], ':', c='gray')
plt.xlabel('predicted'); plt.ylabel('observed pass rate'); plt.title('Calibration (grouped CV)'); plt.show()
print('lowest decile observed %.2f, highest decile observed %.2f' % (cal.obs.iloc[0], cal.obs.iloc[-1]))

## 4. Conclusions

**Short answer: only weakly.** On students the model has never seen, the best model reaches AUC ≈ 0.60 (95% CI 0.59–0.61 over students) versus 0.50 for guessing, and accuracy ≈ 0.60 versus 0.58 for always predicting "fail". That is real but small: the top predicted decile passes about 57% of the time and the bottom decile about 28%. It can rank sessions a little; it is not good enough to make confident per-session calls.

**What the evidence says**
- **Signal comes from two session features.** Longer sessions go with passing (+0.30 log-odds per SD), more hints with failing (−0.22 per SD). Nothing else adds anything reliable.
- **`prior_score` carries almost no information** (coefficient 0.06, flat pass-rate curve, negative permutation importance). That is surprising for a placement test. It may be worth checking how it is stored on the session record (it varies slightly between one student's sessions, which a single placement test shouldn't).
- **`enrol_year` and `school_code` do not help on new students.** Adding them to the logistic model changes nothing, and for boosted trees they lower grouped-CV AUC slightly.
- **Simple beats complex.** Logistic regression on three features matches or beats boosted trees under honest evaluation.

**Why the evaluation design matters.** Students differ in pass rate far more than chance allows (variance 0.125 vs 0.041 expected), and that difference is not explained by these columns. With a shuffled split, boosted trees on all features reach AUC 0.626 vs 0.583 when whole students are held out. That extra is leakage: the model recognises students it has seen. If the real use is predicting for returning students, this gap suggests that per-student history (that student's past pass rate) could be a much stronger feature than anything here. That is untested, so treat it as a lead, not a result.

**Caveats**
- The constant-rate model scores AUC below 0.5 under CV because pooled out-of-fold predictions shift slightly between folds; read it as "no skill".
- Associations are not causal (e.g. students who struggle probably request more hints).
- Sessions are not time-stamped, so I could not test forward-in-time performance. If the model would be used prospectively, that check is still needed.
- Hyperparameters were not tuned; with this little signal, tuning is unlikely to change the conclusion much.